In [2]:
import sys
sys.path.insert(0, "../src")

from tb_explain import adapt, explain

1 - Upper right region with strong TB

In [3]:
tb_output = adapt(
    probabilities={"healthy": 0.05, "sick_non_tb": 0.12, "tb": 0.83},
    detections=[
        ((350, 40, 500, 180), "active_tb", 0.91), 
    ],
    image_size=(512, 512),
)

print(tb_output.model_dump_json(indent=2))

{
  "image_classification": {
    "predicted_label": "tb",
    "probabilities": {
      "healthy": 0.05,
      "sick_non_tb": 0.12,
      "tb": 0.83
    }
  },
  "regions": [
    {
      "type": "active_tb",
      "confidence_band": "high",
      "location": "upper_right"
    }
  ]
}


In [4]:
print(explain(tb_output))

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


The radiological analysis indicates a high probability (83%) of tuberculosis (TB) presence, with a particular high confidence in the upper right region of the image. The likelihood of the image depicting a healthy lung condition or non-TB-related sickness is relatively low (combined probability of 0.17%). These findings suggest active TB involvement, necessitating further clinical correlation and management.


## Case 2 — Ambiguous, multiple regions

Moderate TB probability, two detections at different confidence levels.

In [5]:
ambiguous_output = adapt(
    probabilities={"healthy": 0.15, "sick_non_tb": 0.35, "tb": 0.50},
    detections=[
        ((60,  30, 200, 150), "active_tb",  0.62),  # upper-left, medium confidence
        ((280, 300, 420, 450), "latent_tb", 0.45),  # lower-center, low confidence
    ],
    image_size=(512, 512),
)

print(ambiguous_output.model_dump_json(indent=2))

{
  "image_classification": {
    "predicted_label": "tb",
    "probabilities": {
      "healthy": 0.15,
      "sick_non_tb": 0.35,
      "tb": 0.5
    }
  },
  "regions": [
    {
      "type": "active_tb",
      "confidence_band": "medium",
      "location": "upper_left"
    },
    {
      "type": "latent_tb",
      "confidence_band": "low",
      "location": "lower_right"
    }
  ]
}


In [6]:
print(explain(ambiguous_output))

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The TB detection model has identified a likelihood of tuberculosis (tb) in the patient's chest X-ray, with a high probability of 50%. There is a medium confidence region of active tuberculosis located in the upper left area of the lung. Additionally, a lower confidence indication of latent tuberculosis is observed in the lower right lung area. Further clinical evaluation and possibly additional testing are recommended due to these findings.


## Case 3 — Healthy, no detections

In [7]:
healthy_output = adapt(
    probabilities={"healthy": 0.91, "sick_non_tb": 0.07, "tb": 0.02},
    detections=[],
    image_size=(512, 512),
)

print(explain(healthy_output))

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The computed tomography (CT) scan analysis indicates a high probability (91%) of the lung being healthy, with minimal likelihood (2%) of tuberculosis (TB) presence, and a very low probability (7%) of the lung being sick without TB. No specific regions of concern were identified in the scan.
